In [1]:
# scraping.ipynb
# ============================================================
# TAHAP 1: SCRAPING DATA DARI GOOGLE PLAY STORE
# ============================================================

# Install library yang dibutuhkan
!pip install google-play-scraper pandas

from google_play_scraper import reviews, Sort
import pandas as pd
import time

# ============================================================
# Konfigurasi target aplikasi
# Contoh: scraping ulasan aplikasi Gojek
# ============================================================

APP_ID = 'com.gojek.app'   # Ganti sesuai target
LANG   = 'id'              # Bahasa Indonesia
COUNT  = 5000              # Ambil 5000 ulasan (melebihi minimum)

print(f"Mulai scraping ulasan dari: {APP_ID}")
print(f"Target jumlah data: {COUNT}")

# ============================================================
# Proses Scraping
# ============================================================
result, continuation_token = reviews(
    APP_ID,
    lang=LANG,
    country='id',
    sort=Sort.NEWEST,
    count=COUNT,
)

# Konversi ke DataFrame
df = pd.DataFrame(result)
print(f"\n✅ Berhasil scraping {len(df)} ulasan")
print(df.head())

# ============================================================
# Pilih kolom yang relevan
# ============================================================
df = df[['userName', 'content', 'score', 'at']]
df.columns = ['username', 'review', 'rating', 'date']

# Hapus duplikat dan baris kosong
df.drop_duplicates(subset='review', inplace=True)
df.dropna(subset=['review'], inplace=True)

print(f"\n📊 Total data setelah cleaning: {len(df)}")
print(df['rating'].value_counts())

# ============================================================
# Simpan dataset
# ============================================================
df.to_csv('dataset.csv', index=False, encoding='utf-8-sig')
print("\n✅ Dataset berhasil disimpan ke dataset.csv")
df.head()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00
Mulai scraping ulasan dari: com.gojek.app
Target jumlah data: 5000

✅ Berhasil scraping 5000 ulasan
                               reviewId         userName  \
0  e4e6e922-f5f4-4d51-82ea-f17f2e779aef  Pengguna Google   
1  683dea64-361e-40b5-987f-57896d4d24d9  Pengguna Google   
2  60f19aa8-25d7-47aa-b3df-cfd4ed2fa898  Pengguna Google   
3  e977aa44-6c04-4c27-8f19-c95d4a2135fc  Pengguna Google   
4  84c5361a-b657-4025-be16-401a4c830013  Pengguna Google   

                                           userImage  \
0  https://play-lh.googleusercontent.com/EGemoI2N...   
1  https://play-lh.googleusercontent.com/EGemoI2N...   
2  https://play-lh.googleusercontent.com/EGemoI2N...   
3  https://play-lh.googleusercontent.com/EGemoI2N...   
4  https://play-lh.googleusercontent.com/EGemoI2N...   

                                             content  score  thumbsUpCount  \
0  lucu ya. dia punya fitur pengiriman prior

,username,review,rating,date
0,Pengguna Google,"lucu ya. dia punya fitur pengiriman prioritas,...",1,2026-04-13 11:28:24
1,Pengguna Google,"jele bgt driver""nya sekarang, pesen prioritas+...",1,2026-04-13 11:19:33
2,Pengguna Google,alhamdulillah nyaman tiba2 udh kluar tol mksh,5,2026-04-13 10:59:38
3,Pengguna Google,sangat bagus,5,2026-04-13 10:58:10
4,Pengguna Google,ya enak,5,2026-04-13 10:47:12


In [2]:
# ============================================================
# TAHAP 1B: SCRAPING TAMBAHAN (GRAB & MAXIM)
# ============================================================
from google_play_scraper import reviews, Sort
import pandas as pd

# Target aplikasi tambahan
apps = {
    'Grab': 'com.grabtaxi.passenger',
    'Maxim': 'com.taxsee.taxsee'
}
COUNT = 4000  # Ambil 4000 data dari masing-masing aplikasi
all_new_reviews = []

for name, app_id in apps.items():
    print(f"⏳ Mulai scraping ulasan dari: {name} ({app_id})...")
    result, _ = reviews(
        app_id,
        lang='id',
        country='id',
        sort=Sort.NEWEST,
        count=COUNT,
    )
    df_temp = pd.DataFrame(result)
    df_temp['app'] = name
    all_new_reviews.extend(df_temp.to_dict('records'))
    print(f"✅ {name} selesai! Dagingkan {len(result)} ulasan.")

# Konversi ke DataFrame
df_new = pd.DataFrame(all_new_reviews)

# Pilih kolom yang relevan & ubah nama kolom
df_new = df_new[['userName', 'content', 'score', 'at', 'app']]
df_new.columns = ['username', 'review', 'rating', 'date', 'app']

# Hapus duplikat dan baris kosong
df_new.drop_duplicates(subset='review', inplace=True)
df_new.dropna(subset=['review'], inplace=True)

# Labeling data
def labeling(rating):
    if rating <= 2:
        return 'negatif'
    elif rating == 3:
        return 'netral'
    else:
        return 'positif'

df_new['label'] = df_new['rating'].apply(labeling)

# ============================================================
# PENGGABUNGAN DATA
# ============================================================
# Baca data gojek dari Tahap 1 (dataset.csv)
df_gojek = pd.read_csv('dataset.csv')
df_gojek['app'] = 'Gojek'
# Labeling data gojek
df_gojek['label'] = df_gojek['rating'].apply(labeling)

# Gabungkan semua data
df_gabungan = pd.concat([df_gojek, df_new], ignore_index=True)
print(f"\n📊 Total data gabungan (Gojek + Grab + Maxim): {len(df_gabungan)}")

# Simpan dataset gabungan
df_gabungan.to_csv('dataset_gabungan.csv', index=False)
print("✅ Dataset gabungan berhasil disimpan ke dataset_gabungan.csv")
df_gabungan.head()


⏳ Mulai scraping ulasan dari: Grab (com.grabtaxi.passenger)...
✅ Grab selesai! Dagingkan 4000 ulasan.
⏳ Mulai scraping ulasan dari: Maxim (com.taxsee.taxsee)...
✅ Maxim selesai! Dagingkan 4000 ulasan.

📊 Total data gabungan (Gojek + Grab + Maxim): 9402
✅ Dataset gabungan berhasil disimpan ke dataset_gabungan.csv


,username,review,rating,date,app,label
0,Pengguna Google,"lucu ya. dia punya fitur pengiriman prioritas,...",1,2026-04-13 11:28:24,Gojek,negatif
1,Pengguna Google,"jele bgt driver""nya sekarang, pesen prioritas+...",1,2026-04-13 11:19:33,Gojek,negatif
2,Pengguna Google,alhamdulillah nyaman tiba2 udh kluar tol mksh,5,2026-04-13 10:59:38,Gojek,positif
3,Pengguna Google,sangat bagus,5,2026-04-13 10:58:10,Gojek,positif
4,Pengguna Google,ya enak,5,2026-04-13 10:47:12,Gojek,positif
